In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
import ifcopenshell.geom
import pandas as pd
from damage_generator import  IfcQuery, ElementDamageClustering, pset_epoch_id, DamageCreation, stable_damage_guid
import numpy as np
from rtree import index
from scipy.spatial import cKDTree
from host_assignment import DamageHostAssignment
from sklearn.decomposition import PCA
import alphashape
from shapely.geometry import Polygon, MultiPoint
from shapely.ops import unary_union
import trimesh
import ifcopenshell.api as api
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D
import ifcopenshell.util.placement as plc
from damage_to_host_projection import verts_faces_to_ifc, wall_mesh_vertices, wall_face_plane_from_placement, closest_wall_face_plane_to_patch, project_points_to_plane
import open3d as o3d

# 1) Dataset preprocessing and enrichment

In [ ]:
original_pc = pd.read_csv(r"data/input/point_cloud_classified.csv", sep=';').drop(columns=['host id']).rename(columns={'//X': 'X', 'Scalar field': 'Classification'})
original_pc.head()

In [ ]:
# Add the temporal information (DateTime survey event)
pc = original_pc.copy()
pc['Timestamp'] = '2025-12-17 09:00'
pc.head()

In [ ]:
pc['Classification'].value_counts()

In [ ]:
# Remove unwanted damages
pc = pc[(pc['Classification'] != 0) & (pc['Classification'] != 5)]
# pc['Classification'].value_counts()

In [ ]:
# Map damage name to the damage class
class_to_damage = {
    1: "Bacteria",
    2: "Lichen",
    3: "Moss",
    4: "Plant"
}

pc['damage_type'] = pc['Classification'].map(class_to_damage)

In [ ]:
model = ifcopenshell.open(r"data/input/UMINHO_monastery_Junqueria_2.ifc")

In [ ]:
settings = ifcopenshell.geom.settings()
settings.set(settings.USE_WORLD_COORDS, True)

In [ ]:
p = index.Property()
p.dimension = 3
idx = index.Index(properties=p)

In [ ]:
# Merge the ifc hosting elements
host_elements = (model.by_type('IfcColumn') + model.by_type('IfcWall') + model.by_type('IfcBuildingElementProxy') + model.by_type('IfcSlab') + model.by_type('IfcCovering'))
# print(len(host_elements))

In [ ]:
for i in host_elements:
    print(i)

In [ ]:
# Define the point cloud as a numpy array and prepare an array to store the assigned host element for each point
pc_xyz_np = pc[['X', 'Y', 'Z']].to_numpy()
assigned_host_id = np.empty(len(pc_xyz_np), dtype=object)

In [ ]:
# Divide the point cloud in chunks and assign to each point the corresponding element
chunk = 50000
dha = DamageHostAssignment(settings, model, idx, host_elements)
dha.elements_bounding_box_tolerance()
dha.kdt_for_elements()

for s in range(0, len(pc_xyz_np), chunk):
    assigned_host_id[s:s+chunk] = dha.assign_host(pc_xyz_np[s:s+chunk], max_distance=2)

In [ ]:
pc.info()

In [ ]:
# Add the host id to the dataframe 
pc["host_id"] = assigned_host_id

print(pc["host_id"].isna().sum())
print(pc["host_id"].value_counts(dropna=False))

In [ ]:
rejected = pc[pc["host_id"].isna()]

In [ ]:
rejected

In [ ]:
rejected.to_csv(r"data/output/rejected_point_cloud_3.csv", index=False)

In [ ]:

pc = pc[pc["host_id"].notna()]

In [ ]:
pc.info()

In [ ]:
pc.head()

In [ ]:
# guid = '2$xaf8i4P8RfwvrIilvccU'
# element = find_element_by_guid(guid, model)
# element.is_a()

In [ ]:
# Add the ifc element to the dataframe
pc.loc[:, 'ifc_type'] = pc['host_id'].map(lambda g: IfcQuery(g, model).guid_to_ifc_type())
# pc.head()

In [ ]:
# Add 2D or 3D dimensionality to the elements
# pc.loc[:, 'clustering_space'] = pc['ifc_type'].map(lambda g: clustering_space(g))
# pc.head()

In [ ]:
# pc.info()

In [ ]:
# pc['clustering_space'].value_counts()

In [ ]:
# Add the DBSCAN clustering of the damages
# df_2d = pc['clustering_space'].iloc[0] == '2D'
# 
# final_cluster = cluster_2d if df_2d else cluster_3d
# pc = final_cluster(pc)
damage = ElementDamageClustering(pc)
pc = damage.cluster_3d()

In [ ]:
damage

In [ ]:
pc

In [ ]:
previous_enriched_pc = pd.read_csv(r'data/output/enriched_point_cloud.csv')
previous_enriched_pc

In [ ]:
# save the enriched dataframe for later use in .csv format
pc.to_csv(r"data/output/enriched_point_cloud.csv", index=False)

In [ ]:
value = pc.loc[pc['cluster_id'] == 4, 'host_id']
value.value_counts()

In [ ]:
pc['cluster_id'].value_counts()

In [ ]:
for cluster, n in pc['cluster_id'].value_counts().items():
    print(cluster)

In [ ]:
clusters = damage.cluster_properties()
damage_csv = pd.DataFrame(clusters)
damage_csv['damage_id'] = damage_csv.apply(lambda row: stable_damage_guid(
    row['host_id'],
    row['damage_type'],
    row['centroid']
), axis=1)

In [ ]:
damage_csv

In [ ]:
# save the enriched dataframe for later use in .csv format
damage_csv.to_csv(r"data/output/cluster_attributes_2.csv", index=False)

# 1b) Damage clustering visualization

In [ ]:
#read the enriched dataframe for later use in .csv format
pc_visual = pd.read_csv(r"data/output/enriched_point_cloud.csv")
pc_visual

In [ ]:
xyz_coordinates = pc_visual[['X', 'Y', 'Z']].to_numpy()
rgb_colors = pc_visual[['Rf', 'Gf', 'Bf']].to_numpy()

In [ ]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz_coordinates)
pcd.colors = o3d.utility.Vector3dVector(rgb_colors)

In [ ]:
o3d.visualization.draw_geometries([pcd])

# 2) Damage geometry modelling and projection (skippa sta parte)

In [ ]:
# pc_for_geometry_modeling = pc.copy()

In [ ]:
for i, k in pc['cluster_id'].value_counts().items():
    pc_geometry = pc[pc['cluster_id'] == i]
    
    points = pc_geometry[['X', 'Y', 'Z']].to_numpy()
    
    centroid = points.mean(axis=0)
    
    rel = points - centroid
    pca = PCA(n_components=3).fit(rel)
    axes = pca.components_
    
    normal = axes[2]
    tangent_u = axes[0]
    tangent_v = axes[1]
    
    x = rel.dot(tangent_u)
    y = rel.dot(tangent_v)
    
    points_2d = np.vstack((x, y)).T
    mp = MultiPoint(points_2d)
    
    alpha = 4
    shape = alphashape.alphashape(points_2d, alpha)
    
    offset=-0.005
    normal = normal/np.linalg.norm(normal)
    
    vertices_2d, faces = trimesh.creation.triangulate_polygon(shape, engine='triangle')
    vertices_2d = np.array(vertices_2d)
    faces = np.array(faces)
    verts3d_full = centroid + np.outer(vertices_2d[:,0], tangent_u) + np.outer(vertices_2d[:,1], tangent_v) + offset * normal
    
    verts = [tuple(v.tolist()) for v in verts3d_full]
    faces_idx = [tuple(face.tolist()) for face in faces]
    
    host_id = pc.loc[pc['cluster_id'] == i, 'host_id'].value_counts()
    
    for id, count in host_id.items():
        host_element = IfcQuery(id, model).find_element_by_guid()
        pset_name = pset_epoch_id(pc.loc[pc['cluster_id'] == i, 'Timestamp'].iloc[0])
        damage_type = pc.loc[pc['cluster_id'] == i, 'damage_type'].iloc[0]
        timestamp = pc.loc[pc['cluster_id'] == i, 'Timestamp'].iloc[0]
        damage_prova = DamageCreation(model).ingest_single_damage(host_element, damage_type, pset_name, timestamp, id, centroid) #controlla se ha va bene fare damage_prova=
        vertices_list, faces_list = verts_faces_to_ifc(verts, faces_idx)
        
        def hosting_face_plane_for_patch(ifc_wall, V_patch, plc, settings=None):
            wall_verts = wall_mesh_vertices(ifc_wall, settings=settings)
            plane1, plane2 = wall_face_plane_from_placement(ifc_wall, wall_verts, plc)
            return closest_wall_face_plane_to_patch(V_patch, plane1, plane2)
        
        V = np.array(vertices_list, float)
        plane_point, plane_normal = hosting_face_plane_for_patch(host_element, V, plc, settings=settings)[0]
        V_snapped = project_points_to_plane(V, plane_point, plane_normal, offset=-0.005)
        vertices_list = V_snapped.tolist()
        
        damage_prova.ObjectPlacement = None

        faces_1based = [[int(a)+1, int(b)+1, int(c)+1] for a,b,c in faces_list]
        
        cpl = model.create_entity("IfcCartesianPointList3D", CoordList=vertices_list)
        
        tfs = model.create_entity("IfcTriangulatedFaceSet", Coordinates=cpl, CoordIndex=faces_1based, Closed=False)
        
        geom_ctx = model.by_type("IfcGeometricRepresentationContext")[0]
        
        shape_rep = model.create_entity(
            "IfcShapeRepresentation",
            ContextOfItems=geom_ctx,
            RepresentationIdentifier="Body",
            RepresentationType="TriangulatedFaceSet",
            Items=[tfs]
        )
        
        pdef = model.create_entity("IfcProductDefinitionShape", Representations=[shape_rep])
        damage_prova.Representation = pdef
        

In [ ]:
#chatgpt
def hosting_face_plane_for_patch(ifc_wall, V_patch, plc, settings=None):
    wall_verts = wall_mesh_vertices(ifc_wall, settings=settings)
    plane1, plane2 = wall_face_plane_from_placement(ifc_wall, wall_verts, plc)
    return closest_wall_face_plane_to_patch(V_patch, plane1, plane2)


def alpha_shape_to_single_polygon(points_2d, alpha=4):
    shape = alphashape.alphashape(points_2d, alpha)

    if shape.is_empty:
        return None

    if shape.geom_type == "Polygon":
        return shape

    if shape.geom_type == "MultiPolygon":
        # Keep the largest connected component
        return max(shape.geoms, key=lambda g: g.area)

    return None

In [ ]:
#chatgpt
damage_builder = DamageCreation(model)

for cluster_id, cluster_count in pc["cluster_id"].value_counts().items():
    pc_cluster = pc[pc["cluster_id"] == cluster_id]

    points = pc_cluster[["X", "Y", "Z"]].to_numpy()
    if len(points) < 3:
        print(f"Cluster {cluster_id}: skipped, fewer than 3 points")
        continue

    centroid = points.mean(axis=0)

    rel = points - centroid
    pca = PCA(n_components=3).fit(rel)
    axes = pca.components_

    tangent_u = axes[0]
    tangent_v = axes[1]
    normal = axes[2]

    normal_norm = np.linalg.norm(normal)
    if normal_norm == 0:
        print(f"Cluster {cluster_id}: skipped, zero normal")
        continue
    normal = normal / normal_norm

    x = rel.dot(tangent_u)
    y = rel.dot(tangent_v)
    points_2d = np.column_stack((x, y))

    alpha = 4
    shape = alpha_shape_to_single_polygon(points_2d, alpha=alpha)
    if shape is None:
        print(f"Cluster {cluster_id}: skipped, invalid alpha shape")
        continue

    try:
        vertices_2d, faces = trimesh.creation.triangulate_polygon(shape, engine="triangle")
    except Exception as e:
        print(f"Cluster {cluster_id}: triangulation failed -> {e}")
        continue

    vertices_2d = np.asarray(vertices_2d, dtype=float)
    faces = np.asarray(faces, dtype=int)

    if len(vertices_2d) == 0 or len(faces) == 0:
        print(f"Cluster {cluster_id}: skipped, empty triangulation")
        continue

    offset = -0.005
    verts3d_full = (
        centroid
        + np.outer(vertices_2d[:, 0], tangent_u)
        + np.outer(vertices_2d[:, 1], tangent_v)
        + offset * normal
    )

    verts = [tuple(v) for v in verts3d_full.tolist()]
    faces_idx = [tuple(f) for f in faces.tolist()]

    # Use most frequent host in the cluster
    dominant_host_id = pc_cluster["host_id"].value_counts().idxmax()
    host_element = IfcQuery(dominant_host_id, model).find_element_by_guid()

    if host_element is None:
        print(f"Cluster {cluster_id}: skipped, host element not found")
        continue

    timestamp = pc_cluster["Timestamp"].iloc[0]
    damage_type = pc_cluster["damage_type"].iloc[0]
    pset_name = pset_epoch_id(timestamp)

    try:
        damage_prova = damage_builder.ingest_single_damage(
            host_element,
            damage_type,
            pset_name,
            timestamp,
            dominant_host_id,
            centroid
        )
    except Exception as e:
        print(f"Cluster {cluster_id}: damage creation failed -> {e}")
        continue

    if damage_prova is None:
        print(f"Cluster {cluster_id}: ingest_single_damage returned None")
        continue

    try:
        vertices_list, faces_list = verts_faces_to_ifc(verts, faces_idx)

        V = np.array(vertices_list, dtype=float)
        plane_result = hosting_face_plane_for_patch(
            host_element,
            V,
            plc,
            settings=settings
        )

        plane_point, plane_normal = plane_result[0]
        V_snapped = project_points_to_plane(V, plane_point, plane_normal, offset=-0.005)
        vertices_list = V_snapped.tolist()

        faces_1based = [[int(a) + 1, int(b) + 1, int(c) + 1] for a, b, c in faces_list]

        cpl = model.create_entity("IfcCartesianPointList3D", CoordList=vertices_list)

        tfs = model.create_entity(
            "IfcTriangulatedFaceSet",
            Coordinates=cpl,
            CoordIndex=faces_1based,
            Closed=False
        )

        geom_ctx = model.by_type("IfcGeometricRepresentationContext")[0]

        shape_rep = model.create_entity(
            "IfcShapeRepresentation",
            ContextOfItems=geom_ctx,
            RepresentationIdentifier="Body",
            RepresentationType="TriangulatedFaceSet",
            Items=[tfs]
        )

        pdef = model.create_entity(
            "IfcProductDefinitionShape",
            Representations=[shape_rep]
        )

        # Optional: only do this if you know your workflow needs it
        damage_prova.ObjectPlacement = None
        damage_prova.Representation = pdef

    except Exception as e:
        print(f"Cluster {cluster_id}: geometry assignment failed -> {e}")
        continue

In [ ]:
model.write(r"data/output/ifc_with_geom_damage_4.ifc")

In [ ]:
pc_geometry = pc[pc['cluster_id'] == 199]
pc_geometry

In [ ]:
pc_geometry.info()

In [ ]:
points = pc_geometry[['X', 'Y', 'Z']].to_numpy()
points

In [ ]:
# Compute the centroid of the point cloud cluster
centroid = points.mean(axis=0)
centroid

In [ ]:
# Compute the PCA to determine the size of the scatter in each direction
rel = points - centroid
pca = PCA(n_components=3).fit(rel)
axes = pca.components_
rel

In [ ]:
%matplotlib tk
# Visual check by plotting the point cloud and the PCA axes

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(points[:,0], points[:,1], points[:,2], alpha=0.2)
ax.scatter(*centroid, color='red', s=50)

data_range = np.ptp(points, axis=0).max()
axis_length = data_range * 0.6

for i, c in enumerate(['b','orange','g']):
    start = centroid - axes[i]*axis_length
    end = centroid + axes[i]*axis_length
    ax.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], color=c, linewidth=3)

ax.set_box_aspect([1,1,1])
plt.show()

In [ ]:
# Assign a name to each of the axes
normal = axes[2]
tangent_u = axes[0]
tangent_v = axes[1]

In [ ]:
# Determine the dot product of each point and the tangents (u, v) to transform the PC in a 2D shape
x = rel.dot(tangent_u)
y = rel.dot(tangent_v)
print(len(x))

In [ ]:
# 2D PC array
points_2d = np.vstack((x, y)).T
points_2d

In [ ]:
mp = MultiPoint(points_2d)
mp

In [ ]:
# Create a bounding polygons for the damage pc
alpha = 4
# if alpha is None:
#     alpha = alphashape.optimizealpha(pts2d)

shape = alphashape.alphashape(points_2d, alpha)
shape

In [ ]:
# simplify_tol = 0.01
# # if hasattr(shape, 'simplify'):
# shape = shape.simplify(simplify_tol, preserve_topology = True)

In [ ]:
offset=-0.005

In [ ]:
# Extract the coordinates of the points defining the perimeter of the polygon
# exterior = np.array(shape.exterior.coords)
# exterior

In [ ]:
# verts3d = centroid + np.outer(exterior[:,0], tangent_u) + np.outer(exterior[:,1], tangent_v)
# verts3d

In [ ]:
# Compute the vector perpendicular to u and v
# normal_3d = np.cross(tangent_u, tangent_v)

In [ ]:
# Normalize the vector (length = 1)
normal = normal/np.linalg.norm(normal)

In [ ]:
# Add an offset in the direction of the normal 
# verts3d = verts3d + offset*normal_3d
# verts3d

In [ ]:
# Create a triangulated mesh from the polygon
# poly2d = trimesh.path.polygons.to_polygon(exterior)  # fallback
vertices_2d, faces = trimesh.creation.triangulate_polygon(shape, engine='triangle')
vertices_2d = np.array(vertices_2d)
faces = np.array(faces)

In [ ]:
# Transform the 2D vertices of the triangulated mesh back to 3D and add the offset in the normal direction
verts3d_full = centroid + np.outer(vertices_2d[:,0], tangent_u) + np.outer(vertices_2d[:,1], tangent_v) + offset * normal

In [ ]:
verts3d_full

In [ ]:
# Convert the vertices and faces to lists of tuples for IFC representation
verts = [tuple(v.tolist()) for v in verts3d_full]
faces_idx = [tuple(face.tolist()) for face in faces]

In [ ]:
verts

In [ ]:
# vertices = verts3d_full.tolist()
# vertices

In [ ]:
# faces_list = faces.tolist()
# faces_list

In [ ]:
# print("type(vertices):", type(verts))
# print("len(vertices):", len(verts))
# print("vertices sample (first 5):", verts[:5])
# 
# print("type(faces):", type(faces_idx))
# print("len(faces):", len(faces_idx))
# print("faces sample (first 5):", faces_idx[:5])
# 
# # For nested structures, also inspect element types:
# if len(verts)>0:
#     print("vertex[0] type, len (if sequence):", type(verts[0]), getattr(verts[0], "__len__", lambda: None)())
# if len(faces_idx)>0:
#     print("face[0] type, len (if sequence):", type(faces_idx[0]), getattr(faces_idx[0], "__len__", lambda: None)())


In [ ]:
host_element = IfcQuery('2$xaf8i4P8RfwvrIilvclo', model).find_element_by_guid()
host_element

In [ ]:
damage_prova = model.create_entity(
    "IfcCovering",
    GlobalId=stable_damage_guid('2$xaf8i4P8RfwvrIilvclo', 'Lichen', centroid),
    # OwnerHistory=model.by_type("IfcOwnerHistory")[0],
    Name="Biological_lichen",
    PredefinedType="USERDEFINED"
)

In [ ]:
damage_prova

In [ ]:
verts

In [ ]:
# Convert vertices and faces to the format required by IFC (lists of tuples)
verts = np.asarray(verts)
faces_arr = np.asarray(faces_idx)

vertices_list = [[float(x), float(y), float(z)] for x, y, z in verts]

faces_list = []
for f in faces_arr:
    # ensure length-3 face
    if len(f) != 3:
        raise ValueError("Faces must be triangles (length 3). Got: %r" % (f,))
    # convert to ints
    tri = [int(f[0]), int(f[1]), int(f[2])]
    faces_list.append(tri)
    

In [ ]:
vertices_list

In [ ]:
# M = np.array(plc.get_local_placement(damage_prova.ObjectPlacement), dtype=float)

In [ ]:
# M

In [ ]:
def wall_mesh_vertices(ifc_wall, settings=None):
    """
    Returns Nx3 numpy array of wall mesh vertices in WORLD coordinates.
    """
    if settings is None:
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)  # important!
        # optional:
        # settings.set(settings.DISABLE_OPENING_SUBTRACTIONS, True)

    shape = ifcopenshell.geom.create_shape(settings, ifc_wall)
    verts = np.array(shape.geometry.verts, dtype=float).reshape((-1, 3))
    return verts

In [ ]:
# Define a function to get the plane of the wall face from its placement (origin and normal)

# def wall_face_plane_from_placement(wall):
#     
#     M = np.array(plc.get_local_placement(wall.ObjectPlacement), dtype=float)  # 4x4
#     origin = M[:3, 3]
#     x = M[:3, 0]  # local X in world
#     z = M[:3, 2]  # local Z in world
#     
#     y = np.cross(z, x)  # local Y (normal)
#     y = y / np.linalg.norm(y)
#     
#     return origin, y

In [ ]:
# Define a function to get the plane of the wall face from its placement (origin and normal)

def wall_face_plane_from_placement(wall, wall_verts, plc):
    """
    Returns two planes (p_min, n) and (p_max, n) where n is the wall side normal,
    and p_min/p_max are points lying on the two opposite side faces.
    """
    
    M = np.array(plc.get_local_placement(wall.ObjectPlacement), dtype=float)  # 4x4
    origin = M[:3, 3]
    x = M[:3, 0]  # local X in world
    x = x / np.linalg.norm(x)
    z = M[:3, 2]  # local Z in world
    z = M[:3, 2]; z = z / np.linalg.norm(z)
      
    y = np.cross(z, x)  # local Y (normal)
    y = y / np.linalg.norm(y)
    
    # signed distances of wall vertices along y, relative to origin
    d = (wall_verts - origin) @ y
    dmin, dmax = float(d.min()), float(d.max())

    p_min = origin + y * dmin
    p_max = origin + y * dmax
    return (p_min, y), (p_max, y)

In [ ]:
def closest_wall_face_plane_to_patch(V_patch, planeA, planeB):
    """
    planeA, planeB: (p, n) with n unit
    Returns the plane (p, n) closer to the patch vertices V_patch.
    """
    pA, nA = planeA
    pB, nB = planeB

    # signed distances
    dA = (V_patch - pA) @ nA
    dB = (V_patch - pB) @ nB

    # use mean absolute distance as "closeness" score
    scoreA = np.mean(np.abs(dA))
    scoreB = np.mean(np.abs(dB))

    return planeA, nA if scoreA <= scoreB else planeB, nB

In [ ]:
def hosting_face_plane_for_patch(ifc_wall, V_patch, plc, settings=None):
    wall_verts = wall_mesh_vertices(ifc_wall, settings=settings)
    plane1, plane2 = wall_face_plane_from_placement(ifc_wall, wall_verts, plc)
    return closest_wall_face_plane_to_patch(V_patch, plane1, plane2)

In [ ]:
# project the vertices of the damage patch to the plane of the wall face, with a small offset in the normal direction to avoid z-fighting in visualization

def project_points_to_plane(V, plane_point, plane_normal, offset=-0.005):
    """
    V: (N,3) vertices
    plane_point: (3,)
    plane_normal: (3,) unit normal
    offset: meters along normal (positive = away from surface)
    """
    # n = plane_normal / np.linalg.norm(plane_normal)
    n = plane_normal
    # signed distance from plane
    d = (V - plane_point) @ n
    V_proj = V - np.outer(d, n)
    return V_proj + offset * n

V = np.array(vertices_list, float)

# plane_point = V.mean(axis=0)     # PCA surface point 
# plane_normal = normal            # from PCA plane (damage patch)

# plane_point, plane_normal = wall_face_plane_from_placement(damage_prova)

plane_point, plane_normal = hosting_face_plane_for_patch(host_element, V, plc, settings=settings)[0]

V_snapped = project_points_to_plane(V, plane_point, plane_normal, offset=-0.005)
vertices_list = V_snapped.tolist()

In [ ]:
print(plane_point)

In [ ]:
V

In [ ]:
plane_normal

In [ ]:
d_before = (V - plane_point) @ plane_normal
d_after  = (np.array(vertices_list) - plane_point) @ plane_normal
print("mean abs dist before:", float(np.mean(np.abs(d_before))))
print("mean abs dist after :", float(np.mean(np.abs(d_after))))

In [ ]:
# print(np.array(vertices_list)[:5])
# print(np.array(vertices_corrected)[:5])

In [ ]:

# # M maps local -> world
# M = np.array(plc.get_local_placement(damage_prova.ObjectPlacement), dtype=float)
# Minv = np.linalg.inv(M)
# 
# Vw = np.array(vertices_list, dtype=float)                 # (N,3) world coords
# Vw_h = np.c_[Vw, np.ones(len(Vw))]                        # (N,4)
# Vl_h = (Minv @ Vw_h.T).T                                  # (N,4)
# vertices_local = Vl_h[:, :3].tolist()   

In [ ]:
# print(damage_prova.__class__)

In [ ]:
# api.run(
#     "geometry.add_mesh_representation",
#     model,
#     context = damage_prova,
#     vertices=verts,
#     faces=faces_idx,
# )

In [ ]:
damage_prova.ObjectPlacement = None

faces_1based = [[int(a)+1, int(b)+1, int(c)+1] for a,b,c in faces_list]

cpl = model.create_entity("IfcCartesianPointList3D", CoordList=vertices_list)

tfs = model.create_entity("IfcTriangulatedFaceSet", Coordinates=cpl, CoordIndex=faces_1based, Closed=False)

geom_ctx = model.by_type("IfcGeometricRepresentationContext")[0]

shape_rep = model.create_entity(
    "IfcShapeRepresentation",
    ContextOfItems=geom_ctx,
    RepresentationIdentifier="Body",
    RepresentationType="TriangulatedFaceSet",
    Items=[tfs]
)

In [ ]:
# print("damage placement:", damage_prova.ObjectPlacement is not None,
#       "host placement:", host_element.ObjectPlacement is not None)

In [ ]:
pdef = model.create_entity("IfcProductDefinitionShape", Representations=[shape_rep])
damage_prova.Representation = pdef

In [ ]:
print("Assigned Representation:", damage_prova.Representation is not None)
print("ShapeRepresentation count in model:", len(model.by_type("IfcShapeRepresentation")))

In [ ]:
model.write(r"data/output/ifc_with_geom_damage_3.ifc")

In [ ]:
# mesh = trimesh.Trimesh(vertices=verts3d_full, faces=faces)
# mesh.export("damage_patch.obj")

In [ ]:
# def centroid(v): return np.mean(v, axis=0)
# 
# print("mesh centroid:", centroid(verts))
# print("wall centroid:", centroid(wall_pts))
# print("centroid distance (m):", np.linalg.norm(centroid(verts)-centroid(wall_pts)))

# 3) Damage attribute assignment and IFC data ingestion

In [ ]:
# before doing this go to vs code and generate the filtered_cluster_attributes.csv file
clusters = damage.cluster_properties()
clusters

In [ ]:
damage_csv = pd.DataFrame(clusters)

In [ ]:
damage_csv

In [ ]:
damage_csv['damage_id'] = damage_csv.apply(lambda row: stable_damage_guid(
    row['host_id'],
    row['damage_type'],
    row['centroid']
), axis=1)
damage_csv

In [ ]:
damage_csv = pd.read_csv(r'data/output/filtered_cluster_attributes_2.csv')
damage_csv

In [ ]:
for datetime in damage_csv['timestamp'].unique().tolist():
    pset_name = pset_epoch_id(datetime)
    damage_csv_updated = DamageCreation(model).ingest_multiple_damage_csv(damage_csv, pset_name, datetime)

In [ ]:
damage_csv_updated 

In [ ]:
model.write(r"data/output/UMINHO_monastery_Junqueria_with_ifc_damages_2.ifc")

In [ ]:
damage_csv_updated.to_csv(r"data/output/filtered_cluster_attributes_2.csv", index=False)

In [ ]:
element = IfcQuery('2$xaf8i4P8RfwvrIilvccU', model).find_element_by_guid()

In [ ]:
for rel in model.by_type('IfcRelAssignsToProduct'):
    if rel.RelatingProduct == element:
        print("The proxy:")
        print(rel.RelatedObjects)
        print("Is related to:")
        print(rel.RelatingProduct)
        print('Through:')
        print(rel,'\n')

In [ ]:
for rel in model.by_type('IfcRelCoversBldgElements'):
    if rel.RelatingBuildingElement == element:
        print("The covering:")
        print(rel.RelatedCoverings)
        print("Is related to:")
        print(rel.RelatingBuildingElement)
        print('Through:')
        print(rel,'\n')